# UPscale — Full Model Architecture (V4)

**AI-powered video restoration & 4× super-resolution** — B.Sc. CS final project, Deep Learning specialization.

This notebook is the **single canonical reference** for the model our project is based on. It contains the complete architecture end-to-end, in the order it was actually built:

| Part | What | Checkpoint produced |
|------|------|---------------------|
| **I** | Shared components — config, BasicVSR + SPyNet model, losses, dataset, metrics, train/eval helpers | — |
| **II** | **V3 base training** — trained from scratch on per-frame synthetic degradation (blur / noise / JPEG + ×4 downscale) | `vsr_model_best.pth` |
| **III** | **V4 YouTube fine-tune** — fine-tunes the V3 checkpoint on real **H.264 codec** degradation (per-clip params + a real encode/decode round-trip), with a replay set against catastrophic forgetting | `vsr_model_v4_best_codec_psnr.pth` |
| **IV** | **Inference & deploy** — real (non-synthetic) video upscaling, the path the production AI service runs |

### The core idea

BasicVSR is a **bidirectional recurrent** video super-resolution network: features from every frame are propagated **backward and forward** along the sequence, aligned frame-to-frame by **optical flow** (SPyNet), then fused and ×4 upsampled with pixel-shuffle. Temporal context is what lets it recover detail a single-image upscaler cannot.

```
LR sequence ─▶ feature extract ─▶ ┌ backward propagation (flow-aligned) ┐
                                  └ forward  propagation (flow-aligned) ┘ ─▶ fuse ─▶ ×4 upsample ─▶ HR sequence
                     ▲
              SPyNet optical flow (pretrained, low-LR fine-tuned)
```

### V3 → V4: why fine-tune

V3 was trained on **per-frame** degradations resampled every frame — it only ever saw independent JPEG/blur/noise, never a real video codec. Real YouTube/WhatsApp footage has **temporally correlated** artifacts: macroblocking, mosquito noise, and ringing from H.264 compression. **V4** fixes the domain gap by fine-tuning V3 on clips passed through a genuine `libx264` encode/decode round-trip, while a replay subset of the old V3 degradation prevents the model from forgetting what it already learned.

> **Runtime note.** This notebook is meant to run in the **GPU training environment** (the same one that ran `Model_v3.ipynb`), not on a laptop. Training/fine-tuning cells load whole clips into RAM and need CUDA. The architecture, degradation, and inference code mirror the production copies in `apps/ai/baseline/` and `scripts/` — keep them in sync (see `apps/ai/AGENTS.md`).

# Part I — Shared components

Everything V3 and V4 have in common: configuration, the model, the losses, the dataset, and the metric/eval helpers. The exact model class here is what ships as `apps/ai/baseline/model_architecture.py` and is loaded by the inference service.

## I.1 — Imports

In [ ]:
from pathlib import Path
import math
import json
import shutil
import random
import zipfile
import hashlib
import subprocess
import tempfile

import cv2
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision.models as models
from torch.utils.data import Dataset, DataLoader

try:
    from skimage.metrics import structural_similarity as ssim_metric
    SKIMAGE_AVAILABLE = True
except Exception:
    SKIMAGE_AVAILABLE = False
    ssim_metric = None

print("torch", torch.__version__, "| cuda available:", torch.cuda.is_available())

## I.2 — Configuration

The **recipe** constants (`SEQ_LEN=15`, `SCALE=4`, feature/block counts) define the architecture and must match the deployed checkpoint. Paths, epochs, and LR are training knobs. `SEED` fixes reproducibility across `random` / `numpy` / `torch`.

In [ ]:
PROJECT_ROOT = Path(".").resolve()

WORK_ROOT = PROJECT_ROOT / "vsr_workspace"
DATA_ROOT = WORK_ROOT / "data"

TRAIN_ZIP_PATH = PROJECT_ROOT / "train_sharp.zip"
VAL_ZIP_PATH = PROJECT_ROOT / "val_sharp.zip"

TRAIN_SHARP_DIR = DATA_ROOT / "train_sharp"
VAL_SHARP_DIR = DATA_ROOT / "val_sharp"

EXPERIMENT_ROOT = WORK_ROOT / "experiments" / "model_v3_seq15"
SAVE_DIR = EXPERIMENT_ROOT / "checkpoints"
EPOCH_CKPT_DIR = SAVE_DIR / "epoch_snapshots"
RESULTS_DIR = EXPERIMENT_ROOT / "results"
LOGS_DIR = EXPERIMENT_ROOT / "logs"

SPLIT_ROOT = WORK_ROOT / "experiments" / "model_v3_seq15" / "splits"

for p in [WORK_ROOT, DATA_ROOT, EXPERIMENT_ROOT, SAVE_DIR, EPOCH_CKPT_DIR, RESULTS_DIR, LOGS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

# ── reproducibility ──────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

# ── runtime ──────────────────────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
PIN_MEMORY = True

# ── architecture recipe (must match the deployed checkpoint) ─────────────────
SEQ_LEN = 15          # frames per sliding window (odd; center frame is the target)
SCALE = 4             # ×4 super-resolution
PATCH_SIZE = 64       # LR training crop (HR crop = PATCH_SIZE * SCALE)

# ── V3 training hyper-parameters ─────────────────────────────────────────────
BATCH_SIZE = 4
NUM_EPOCHS = 150
LR = 2e-4
NUM_WORKERS = 8
WEIGHT_DECAY = 1e-4
GRAD_CLIP_NORM = 1.0

# split / dataset
VAL_TO_VAL_COUNT = 5
VAL_TO_TEST_COUNT = 5
MAX_TRAIN_SEQS = None
MAX_FRAMES_PER_SEQ = 200
RESET_PROCESSED_DATASET = False
FORCE_REBUILD_SPLITS = False

# scheduler
USE_SCHEDULER = True
SCHEDULER_PATIENCE = 5
SCHEDULER_FACTOR = 0.5
MIN_LR = 1e-6

# validation
FAST_VAL_MAX_BATCHES = 20
FULL_VAL_EVERY = 2
FULL_FRAME_EVAL = False
SHAVE_BORDER = SCALE

# persistence
AUTO_RESUME = True
KEEP_EVERY_N_EPOCHS = 10

BEST_PATH = SAVE_DIR / "vsr_model_best_loss.pth"
BEST_PSNR_PATH = SAVE_DIR / "vsr_model_best_psnr.pth"
LAST_PATH = SAVE_DIR / "vsr_model_last.pth"
TRAINING_STATE_PATH = SAVE_DIR / "training_state.pth"
HISTORY_JSON_PATH = LOGS_DIR / "history.json"
TEST_METRICS_JSON_PATH = LOGS_DIR / "test_metrics.json"

print("device:", device)
print("EXPERIMENT_ROOT:", EXPERIMENT_ROOT)
print("SPLIT_ROOT:", SPLIT_ROOT)
print("SEQ_LEN:", SEQ_LEN, "| SCALE:", SCALE, "| PATCH_SIZE:", PATCH_SIZE)

## I.3 — Model architecture: BasicVSR + SPyNet

The network has five parts:

- **`ResidualBlock` / `ConvResidualBlocks`** — the basic conv building block (LeakyReLU, residual add) used everywhere.
- **`SPyNet`** — a Spatial Pyramid Network that estimates dense **optical flow** between two frames. Pretrained on OpenMMLab weights (auto-downloaded on first init), then fine-tuned at a low LR — the paper's recipe.
- **`flow_warp`** — warps a feature map by a flow field via `grid_sample` (NaN-guarded).
- **`PixelShuffleUpsample`** — two ×2 pixel-shuffle stages = ×4, ending in a 3-channel RGB conv.
- **`BasicVSRRecurrentSeq`** — ties it together: extract features → compute forward/backward flows → **backward** recurrent pass → **forward** recurrent pass → fuse both directions → upsample every frame.

> This class is the source of truth for `apps/ai/baseline/model_architecture.py`. **Do not change it without an explicit request** — the shipped checkpoints depend on the exact layer shapes.

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, 3, 1, 1)
        self.conv2 = nn.Conv2d(channels, channels, 3, 1, 1)
        self.act = nn.LeakyReLU(0.1, inplace=True)

    def forward(self, x):
        identity = x
        out = self.act(self.conv1(x))
        out = self.conv2(out)
        return identity + out

class ConvResidualBlocks(nn.Module):
    def __init__(self, in_channels, out_channels, num_blocks):
        super().__init__()
        self.head = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, 1, 1),
            nn.LeakyReLU(0.1, inplace=True)
        )
        self.body = nn.Sequential(*[ResidualBlock(out_channels) for _ in range(num_blocks)])

    def forward(self, x):
        return self.body(self.head(x))


# ===================== SPyNet =====================
class SPyNetBasicModule(nn.Module):
    """A single level of SPyNet."""
    def __init__(self):
        super().__init__()
        self.basic_module = nn.ModuleList([
            self._make_layer(8, 32),
            self._make_layer(32, 64),
            self._make_layer(64, 32),
            self._make_layer(32, 16),
            self._make_layer(16, 2),
        ])

    def _make_layer(self, in_ch, out_ch):
        m = nn.Module()
        m.conv = nn.Conv2d(in_ch, out_ch, 7, 1, 3)
        return m

    def forward(self, tensor_input):
        x = tensor_input
        for i, layer in enumerate(self.basic_module):
            x = layer.conv(x)
            if i < len(self.basic_module) - 1:  # no ReLU on last layer
                x = F.relu(x, inplace=True)
        return x

class SPyNet(nn.Module):
    """SPyNet: Spatial Pyramid Network for optical flow estimation.
    Pretrained weights from: https://github.com/open-mmlab/mmediting
    """
    PRETRAINED_URL = (
        "https://download.openmmlab.com/mmediting/restorers/"
        "basicvsr/spynet_20210409-c6c1bd09.pth"
    )

    def __init__(self, pretrained=True):
        super().__init__()
        self.basic_module = nn.ModuleList([SPyNetBasicModule() for _ in range(6)])

        if pretrained:
            import urllib.request, tempfile, os
            weights_dir = os.path.join(tempfile.gettempdir(), "spynet_weights")
            os.makedirs(weights_dir, exist_ok=True)
            weights_path = os.path.join(weights_dir, "spynet_20210409-c6c1bd09.pth")
            if not os.path.exists(weights_path):
                print(f"Downloading SPyNet weights to {weights_path}...")
                urllib.request.urlretrieve(self.PRETRAINED_URL, weights_path)
                print("Download complete.")
            state_dict = torch.load(weights_path, map_location="cpu")
            self.load_state_dict(state_dict)
            print("Loaded pretrained SPyNet weights")

        self.register_buffer("mean", torch.Tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.Tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

    def preprocess(self, tensor_input):
        return (tensor_input - self.mean) / self.std

    def forward(self, ref, supp):
        """Estimate optical flow from ref to supp."""
        ref = [self.preprocess(ref)]
        supp = [self.preprocess(supp)]

        # build pyramid
        for _ in range(5):
            ref.insert(0, F.avg_pool2d(ref[0], kernel_size=2, stride=2, count_include_pad=False))
            supp.insert(0, F.avg_pool2d(supp[0], kernel_size=2, stride=2, count_include_pad=False))

        flow = ref[0].new_zeros(ref[0].shape[0], 2, ref[0].shape[2], ref[0].shape[3])

        for level in range(len(ref)):
            upsampled_flow = F.interpolate(flow, scale_factor=2, mode="bilinear", align_corners=True) * 2.0 if level > 0 else flow

            # pad if needed
            h, w = ref[level].shape[2:4]
            uh, uw = upsampled_flow.shape[2:4]
            if uh != h or uw != w:
                upsampled_flow = F.interpolate(upsampled_flow, size=(h, w), mode="bilinear", align_corners=True)

            # warp supp
            flow_for_warp = upsampled_flow.permute(0, 2, 3, 1)
            b, fh, fw, _ = flow_for_warp.shape
            yy, xx = torch.meshgrid(torch.arange(fh, device=flow.device, dtype=flow.dtype),
                                     torch.arange(fw, device=flow.device, dtype=flow.dtype), indexing="ij")
            grid = torch.stack((xx, yy), dim=-1).unsqueeze(0).expand(b, -1, -1, -1)
            vgrid = grid + flow_for_warp
            vgrid[..., 0] = 2.0 * vgrid[..., 0] / max(fw - 1, 1) - 1.0
            vgrid[..., 1] = 2.0 * vgrid[..., 1] / max(fh - 1, 1) - 1.0
            warped = F.grid_sample(supp[level], vgrid, mode="bilinear", padding_mode="border", align_corners=True)

            flow_input = torch.cat([ref[level], warped, upsampled_flow], dim=1)
            flow = upsampled_flow + self.basic_module[level](flow_input)

        return flow


# ===================== Flow Warp =====================
def flow_warp(x, flow):
    x = torch.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)
    flow = torch.nan_to_num(flow, nan=0.0, posinf=0.0, neginf=0.0)

    b, c, h, w = x.shape
    yy, xx = torch.meshgrid(torch.arange(h, device=x.device), torch.arange(w, device=x.device), indexing="ij")
    grid = torch.stack((xx, yy), dim=0).float().unsqueeze(0).repeat(b, 1, 1, 1)
    vgrid = grid + flow

    vgrid_x = 2.0 * vgrid[:, 0] / max(w - 1, 1) - 1.0
    vgrid_y = 2.0 * vgrid[:, 1] / max(h - 1, 1) - 1.0
    vgrid_x = torch.clamp(torch.nan_to_num(vgrid_x, nan=0.0, posinf=1.0, neginf=-1.0), -1.0, 1.0)
    vgrid_y = torch.clamp(torch.nan_to_num(vgrid_y, nan=0.0, posinf=1.0, neginf=-1.0), -1.0, 1.0)
    vgrid = torch.stack((vgrid_x, vgrid_y), dim=-1)

    out = F.grid_sample(x, vgrid, mode="bilinear", padding_mode="border", align_corners=True)
    return torch.nan_to_num(out, nan=0.0, posinf=0.0, neginf=0.0)


# ===================== Upsampler =====================
class PixelShuffleUpsample(nn.Module):
    def __init__(self, num_feats, scale):
        super().__init__()
        if scale == 2:
            self.net = nn.Sequential(
                nn.Conv2d(num_feats, num_feats * 4, 3, 1, 1),
                nn.PixelShuffle(2),
                nn.LeakyReLU(0.1, inplace=True),
                nn.Conv2d(num_feats, 3, 3, 1, 1),
            )
        elif scale == 4:
            self.net = nn.Sequential(
                nn.Conv2d(num_feats, num_feats * 4, 3, 1, 1),
                nn.PixelShuffle(2),
                nn.LeakyReLU(0.1, inplace=True),
                nn.Conv2d(num_feats, num_feats * 4, 3, 1, 1),
                nn.PixelShuffle(2),
                nn.LeakyReLU(0.1, inplace=True),
                nn.Conv2d(num_feats, 3, 3, 1, 1),
            )
        else:
            raise ValueError("Only scale=2 or scale=4 supported")

    def forward(self, x):
        return self.net(x)


# ===================== BasicVSR with SPyNet =====================
class BasicVSRRecurrentSeq(nn.Module):
    def __init__(self, seq_len=7, scale=4, num_feats=64, num_extract_blocks=5, num_prop_blocks=20, num_recon_blocks=5):
        super().__init__()
        if seq_len % 2 == 0:
            raise ValueError("seq_len must be odd")

        self.seq_len = seq_len
        self.scale = scale

        self.feat_extractor = ConvResidualBlocks(3, num_feats, num_extract_blocks)

        # SPyNet for optical flow (pretrained, fine-tuned with lower LR)
        self.flow_estimator = SPyNet(pretrained=True)

        self.backward_trunk = ConvResidualBlocks(num_feats * 2, num_feats, num_prop_blocks)
        self.forward_trunk = ConvResidualBlocks(num_feats * 2, num_feats, num_prop_blocks)

        fusion_layers = [
            nn.Conv2d(num_feats * 2, num_feats, 1, 1, 0),
            nn.LeakyReLU(0.1, inplace=True)
        ]
        for _ in range(num_recon_blocks):
            fusion_layers.append(ResidualBlock(num_feats))
        self.fusion = nn.Sequential(*fusion_layers)

        self.upsample = PixelShuffleUpsample(num_feats, scale)

    def compute_flows(self, x):
        b, t, c, h, w = x.shape
        flows_backward = [None] * (t - 1)
        flows_forward = [None] * (t - 1)

        for i in range(t - 1):
            flows_backward[i] = self.flow_estimator(x[:, i], x[:, i + 1])

        for i in range(1, t):
            flows_forward[i - 1] = self.flow_estimator(x[:, i], x[:, i - 1])

        return flows_forward, flows_backward

    def forward(self, x):
        b, t, c, h, w = x.shape
        feats_batch = self.feat_extractor(x.reshape(b * t, c, h, w))
        feats = list(feats_batch.reshape(b, t, -1, h, w).unbind(dim=1))
        flows_forward, flows_backward = self.compute_flows(x)

        backward_feats = [None] * t
        prop = torch.zeros_like(feats[0])

        for i in range(t - 1, -1, -1):
            if i < t - 1:
                prop = flow_warp(prop, flows_backward[i])
            prop = self.backward_trunk(torch.cat([feats[i], prop], dim=1))
            backward_feats[i] = prop

        forward_prop = torch.zeros_like(feats[0])
        outputs = []

        for i in range(t):
            if i > 0:
                forward_prop = flow_warp(forward_prop, flows_forward[i - 1])

            forward_prop = self.forward_trunk(torch.cat([feats[i], forward_prop], dim=1))
            fused = self.fusion(torch.cat([forward_prop, backward_feats[i]], dim=1))
            out = torch.clamp(self.upsample(fused), 0.0, 1.0)
            outputs.append(out)

        return torch.stack(outputs, dim=1)


def build_model():
    """Construct the exact model the checkpoints were trained with."""
    m = BasicVSRRecurrentSeq(
        seq_len=SEQ_LEN, scale=SCALE, num_feats=64,
        num_extract_blocks=5, num_prop_blocks=20, num_recon_blocks=5,
    ).to(device)
    total = sum(p.numel() for p in m.parameters())
    spynet = sum(p.numel() for p in m.flow_estimator.parameters())
    print(f"{m.__class__.__name__}: {total:,} params (SPyNet {spynet:,})")
    return m

## I.4 — Losses

The training objective blends three complementary terms:

- **Charbonnier** — a smooth L1 (robust to outliers); drives PSNR.
- **Sobel edge L1** — matches image gradients; sharpens edges.
- **VGG19 perceptual** — L1 in a frozen ImageNet feature space; recovers texture the pixel loss washes out.

`CombinedRestorationLoss = 1.0·Charbonnier + 0.05·Edge + 0.1·Perceptual`, and it accepts either a single frame or a `(B, T, C, H, W)` sequence (it flattens the time axis for the edge/perceptual terms).

In [ ]:
class CharbonnierLoss(nn.Module):
    def __init__(self, eps=1e-6):
        super().__init__()
        self.eps = eps

    def forward(self, pred, target):
        diff = pred - target
        return torch.sqrt(diff * diff + self.eps * self.eps).mean()

class SobelEdgeLoss(nn.Module):
    def __init__(self):
        super().__init__()
        sobel_x = torch.tensor([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=torch.float32).view(1, 1, 3, 3)
        sobel_y = torch.tensor([[-1, -2, -1], [0, 0, 0], [1, 2, 1]], dtype=torch.float32).view(1, 1, 3, 3)
        self.register_buffer("sobel_x", sobel_x)
        self.register_buffer("sobel_y", sobel_y)

    def _grad(self, x):
        sobel_x = self.sobel_x.to(device=x.device, dtype=x.dtype)
        sobel_y = self.sobel_y.to(device=x.device, dtype=x.dtype)
        grads = []
        for c in range(x.shape[1]):
            xc = x[:, c:c+1]
            gx = F.conv2d(xc, sobel_x, padding=1)
            gy = F.conv2d(xc, sobel_y, padding=1)
            grads.append(torch.sqrt(gx * gx + gy * gy + 1e-6))
        return torch.cat(grads, dim=1)

    def forward(self, pred, target):
        return F.l1_loss(self._grad(pred), self._grad(target))

class VGGPerceptualLoss(nn.Module):
    def __init__(self):
        super().__init__()
        vgg = models.vgg19(weights=models.VGG19_Weights.IMAGENET1K_V1).features
        self.slice1 = nn.Sequential(*[vgg[i] for i in range(9)])
        self.slice2 = nn.Sequential(*[vgg[i] for i in range(9, 27)])
        for param in self.parameters():
            param.requires_grad = False
        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("std",  torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

    def forward(self, pred, target):
        pred = (pred - self.mean) / self.std
        target = (target - self.mean) / self.std
        pred_f1 = self.slice1(pred)
        target_f1 = self.slice1(target)
        pred_f2 = self.slice2(pred_f1)
        target_f2 = self.slice2(target_f1)
        return F.l1_loss(pred_f1, target_f1) + F.l1_loss(pred_f2, target_f2)

class CombinedRestorationLoss(nn.Module):
    def __init__(self, charbonnier_weight=1.0, edge_weight=0.05, perceptual_weight=0.1):
        super().__init__()
        self.charb = CharbonnierLoss(eps=1e-6)
        self.edge = SobelEdgeLoss()
        self.perceptual = VGGPerceptualLoss()
        self.charbonnier_weight = charbonnier_weight
        self.edge_weight = edge_weight
        self.perceptual_weight = perceptual_weight

    def forward(self, pred, target):
        charb_loss = self.charbonnier_weight * self.charb(pred, target)
        if pred.dim() == 5:
            b, t, c, h, w = pred.shape
            pred_4d = pred.reshape(b * t, c, h, w)
            target_4d = target.reshape(b * t, c, h, w)
        else:
            pred_4d = pred
            target_4d = target
        edge_loss = self.edge_weight * self.edge(pred_4d, target_4d)
        perceptual_loss = self.perceptual_weight * self.perceptual(pred_4d, target_4d)
        return charb_loss + edge_loss + perceptual_loss

## I.5 — Metric & tensor helpers

PSNR / SSIM (border-shaved), tensor↔image conversions, bicubic baseline, and sequence augmentation. These are shared by both training stages and by evaluation.

In [ ]:
def img_to_tensor_rgb(img_rgb):
    return torch.from_numpy(img_rgb.transpose(2, 0, 1)).float() / 255.0

def tensor_to_img(t):
    img = t.detach().cpu().float().numpy().transpose(1, 2, 0)
    img = np.nan_to_num(img, nan=0.0, posinf=1.0, neginf=0.0)
    img = (np.clip(img, 0, 1) * 255.0).astype(np.uint8)
    return img

def shave_tensor_border(x, border):
    if border <= 0:
        return x
    return x[..., border:-border, border:-border]

def calc_psnr(pred, target, max_val=1.0, shave_border=0):
    if not torch.isfinite(pred).all() or not torch.isfinite(target).all():
        return float("nan")
    pred = shave_tensor_border(pred, shave_border)
    target = shave_tensor_border(target, shave_border)
    mse = F.mse_loss(pred, target).item()
    if not math.isfinite(mse):
        return float("nan")
    if mse == 0:
        return 100.0
    return 20 * math.log10(max_val / math.sqrt(mse))

def calc_ssim(pred, target, shave_border=0):
    if not SKIMAGE_AVAILABLE:
        return None
    if not torch.isfinite(pred).all() or not torch.isfinite(target).all():
        return None
    pred = shave_tensor_border(pred, shave_border)
    target = shave_tensor_border(target, shave_border)
    pred_img = tensor_to_img(pred.squeeze(0))
    target_img = tensor_to_img(target.squeeze(0))
    try:
        return ssim_metric(pred_img, target_img, channel_axis=2, data_range=255)
    except Exception:
        return None

def calc_ssim_sequence(pred_seq, target_seq, shave_border=0):
    vals = []
    t = pred_seq.shape[1]
    for i in range(t):
        s = calc_ssim(pred_seq[:, i], target_seq[:, i], shave_border=shave_border)
        if s is not None:
            vals.append(s)
    return float(np.mean(vals)) if len(vals) > 0 else None

def bicubic_upsample_sequence(lr_seq, scale):
    b, t, c, h, w = lr_seq.shape
    x = lr_seq.reshape(b * t, c, h, w)
    up = F.interpolate(x, scale_factor=scale, mode="bicubic", align_corners=False)
    up = torch.clamp(up, 0.0, 1.0)
    up = torch.nan_to_num(up, nan=0.0, posinf=1.0, neginf=0.0)
    return up.view(b, t, c, h * scale, w * scale)

def random_augment_sequences(lr_seq, hr_seq):
    if random.random() < 0.5:
        lr_seq = [np.ascontiguousarray(img[:, ::-1, :]) for img in lr_seq]
        hr_seq = [np.ascontiguousarray(img[:, ::-1, :]) for img in hr_seq]
    if random.random() < 0.5:
        lr_seq = [np.ascontiguousarray(img[::-1, :, :]) for img in lr_seq]
        hr_seq = [np.ascontiguousarray(img[::-1, :, :]) for img in hr_seq]
    if random.random() < 0.5:
        lr_seq = [np.ascontiguousarray(np.rot90(img)) for img in lr_seq]
        hr_seq = [np.ascontiguousarray(np.rot90(img)) for img in hr_seq]
    return lr_seq, hr_seq

print("Metric + tensor helpers ready")

## I.6 — Dataset

`CustomVideoDataset` reads pre-built clip folders (`hr_frames/` + `lr_frames/`), and for each clip enumerates every valid **centered `SEQ_LEN`-window**. In training it random-crops + augments; in eval it center-crops (or uses full frames). The **same class** serves V3 (per-frame LR) and V4 (codec LR) — only the LR frames on disk differ.

In [ ]:
def seed_worker(worker_id):
    cv2.setNumThreads(0)  # one OpenCV pool per worker otherwise thrashes CPU/RAM
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

def eval_collate_fn(batch):
    lr_list, hr_list, meta_list = zip(*batch)
    lr_batch = torch.stack(lr_list, dim=0)
    hr_batch = torch.stack(hr_list, dim=0)
    return lr_batch, hr_batch, list(meta_list)


class CustomVideoDataset(Dataset):
    def __init__(self, split_root, seq_len=15, patch_size=64, training=True, scale=4, full_frame_eval=False):
        self.split_root = Path(split_root)
        self.seq_len = seq_len
        self.patch_size = patch_size
        self.training = training
        self.scale = scale
        self.full_frame_eval = full_frame_eval
        self.half = seq_len // 2

        self.samples = []
        self.clips = []

        clip_dirs = sorted([p for p in self.split_root.iterdir() if p.is_dir()])
        for clip_dir in clip_dirs:
            hr_frames = sorted((clip_dir / "hr_frames").glob("*.png"))
            lr_frames = sorted((clip_dir / "lr_frames").glob("*.png"))
            if len(hr_frames) == 0 or len(lr_frames) == 0:
                continue
            if len(hr_frames) != len(lr_frames):
                continue
            if len(hr_frames) < self.seq_len:
                continue
            self.clips.append(clip_dir)
            for center_idx in range(self.half, len(hr_frames) - self.half):
                self.samples.append({
                    "lr_seq_paths": lr_frames[center_idx - self.half:center_idx + self.half + 1],
                    "hr_seq_paths": hr_frames[center_idx - self.half:center_idx + self.half + 1],
                    "clip_name": clip_dir.name,
                    "center_idx": center_idx,
                })

        if len(self.samples) == 0:
            raise RuntimeError(f"No valid samples found in {self.split_root}")
        print(f"{self.split_root.name}: found {len(self.samples)} samples from {len(self.clips)} clips")

    def __len__(self):
        return len(self.samples)

    def _read_image(self, path):
        img = cv2.imread(str(path), cv2.IMREAD_COLOR)
        if img is None:
            raise RuntimeError(f"Failed to read image: {path}")
        return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    def _random_crop_sequences(self, lr_seq, hr_seq):
        h_lr, w_lr = lr_seq[0].shape[:2]
        lr_crop = self.patch_size
        hr_crop = self.patch_size * self.scale
        if h_lr < lr_crop or w_lr < lr_crop:
            raise RuntimeError(f"LR frame too small for crop: got {(h_lr, w_lr)}, need >= {(lr_crop, lr_crop)}")
        top = random.randint(0, h_lr - lr_crop)
        left = random.randint(0, w_lr - lr_crop)
        lr_seq_crop = [img[top:top + lr_crop, left:left + lr_crop, :] for img in lr_seq]
        hr_top, hr_left = top * self.scale, left * self.scale
        hr_seq_crop = [img[hr_top:hr_top + hr_crop, hr_left:hr_left + hr_crop, :] for img in hr_seq]
        return lr_seq_crop, hr_seq_crop

    def _center_crop_sequences(self, lr_seq, hr_seq):
        h_lr, w_lr = lr_seq[0].shape[:2]
        lr_crop = min(self.patch_size, h_lr, w_lr)
        hr_crop = lr_crop * self.scale
        top = (h_lr - lr_crop) // 2
        left = (w_lr - lr_crop) // 2
        lr_seq_crop = [img[top:top + lr_crop, left:left + lr_crop, :] for img in lr_seq]
        hr_top, hr_left = top * self.scale, left * self.scale
        hr_seq_crop = [img[hr_top:hr_top + hr_crop, hr_left:hr_left + hr_crop, :] for img in hr_seq]
        return lr_seq_crop, hr_seq_crop

    def __getitem__(self, idx):
        sample = self.samples[idx]
        lr_seq = [self._read_image(p) for p in sample["lr_seq_paths"]]
        hr_seq = [self._read_image(p) for p in sample["hr_seq_paths"]]

        if self.training:
            lr_seq, hr_seq = self._random_crop_sequences(lr_seq, hr_seq)
            lr_seq, hr_seq = random_augment_sequences(lr_seq, hr_seq)
        elif not self.full_frame_eval:
            lr_seq, hr_seq = self._center_crop_sequences(lr_seq, hr_seq)

        lr_seq = torch.stack([img_to_tensor_rgb(img) for img in lr_seq], dim=0)
        hr_seq = torch.stack([img_to_tensor_rgb(img) for img in hr_seq], dim=0)
        return lr_seq, hr_seq, {"clip_name": sample["clip_name"], "center_idx": sample["center_idx"]}

print("CustomVideoDataset ready")

## I.7 — Train / eval helpers

`train_one_epoch` (with NaN-guarded batches + grad clipping) and `evaluate_model` (loss, PSNR, SSIM, and a bicubic reference for context). Reused verbatim by V3 training; V4 uses its own thin variant later.

In [ ]:
from tqdm import tqdm

def train_one_epoch(model, loader, optimizer, criterion, device, scaler=None, grad_clip_norm=None):
    model.train()
    running_loss = 0.0
    valid_batches = 0
    skipped_batches = 0

    for lr_seq, hr_seq, _ in tqdm(loader, desc="Train", leave=False):
        lr_seq = lr_seq.to(device, non_blocking=True)
        hr_seq = hr_seq.to(device, non_blocking=True)
        if not torch.isfinite(lr_seq).all() or not torch.isfinite(hr_seq).all():
            skipped_batches += 1
            continue

        optimizer.zero_grad(set_to_none=True)
        pred = model(lr_seq)
        if not torch.isfinite(pred).all():
            skipped_batches += 1
            continue
        loss = criterion(pred, hr_seq)
        if not torch.isfinite(loss):
            skipped_batches += 1
            continue

        loss.backward()
        if grad_clip_norm is not None:
            torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip_norm)
        optimizer.step()

        running_loss += loss.item()
        valid_batches += 1

    if skipped_batches > 0:
        print(f"Skipped {skipped_batches} non-finite train batches")
    return running_loss / valid_batches if valid_batches > 0 else float("nan")


@torch.no_grad()
def evaluate_model(model, loader, criterion, device, scale=4, shave_border=0, max_batches=None, compute_ssim=True):
    model.eval()
    total_loss = total_psnr = total_ssim = total_bicubic_psnr = 0.0
    count = ssim_count = 0
    per_clip = {}

    for batch_idx, (lr_seq, hr_seq, meta_list) in enumerate(loader):
        if max_batches is not None and batch_idx >= max_batches:
            break
        lr_seq = lr_seq.to(device, non_blocking=True)
        hr_seq = hr_seq.to(device, non_blocking=True)
        if not torch.isfinite(lr_seq).all() or not torch.isfinite(hr_seq).all():
            continue
        pred = model(lr_seq)
        if not torch.isfinite(pred).all():
            continue
        loss = criterion(pred, hr_seq)
        if not torch.isfinite(loss):
            continue

        pred_psnr = calc_psnr(pred, hr_seq, shave_border=shave_border)
        bicubic = bicubic_upsample_sequence(lr_seq, scale=scale)
        bicubic_psnr = calc_psnr(bicubic, hr_seq, shave_border=shave_border)
        if not math.isfinite(pred_psnr) or not math.isfinite(bicubic_psnr):
            continue

        total_loss += loss.item()
        total_psnr += pred_psnr
        total_bicubic_psnr += bicubic_psnr
        if compute_ssim:
            pred_ssim = calc_ssim_sequence(pred, hr_seq, shave_border=shave_border)
            if pred_ssim is not None:
                total_ssim += pred_ssim
                ssim_count += 1

        clip_name = meta_list[0]["clip_name"]
        c = per_clip.setdefault(clip_name, {"count": 0, "psnr": 0.0, "bicubic_psnr": 0.0})
        c["count"] += 1
        c["psnr"] += pred_psnr
        c["bicubic_psnr"] += bicubic_psnr
        count += 1

    if count == 0:
        return {"loss": float("nan"), "psnr": float("nan"), "ssim": None, "bicubic_psnr": float("nan"), "per_clip": {}}
    for name in per_clip:
        n = per_clip[name]["count"]
        per_clip[name]["psnr"] /= n
        per_clip[name]["bicubic_psnr"] /= n
    return {
        "loss": total_loss / count,
        "psnr": total_psnr / count,
        "ssim": (total_ssim / ssim_count) if ssim_count > 0 else None,
        "bicubic_psnr": total_bicubic_psnr / count,
        "per_clip": per_clip,
    }

print("Train / eval helpers ready")

# Part II — V3 base training

V3 is trained **from scratch** on the REDS dataset. Each sharp HR frame is degraded **per-frame** into an LR frame (parameters resampled every frame), then the model learns to invert the degradation with temporal context. Output: `vsr_model_best_psnr.pth` → shipped as `apps/ai/checkpoints/vsr_model_best.pth`.

## II.1 — Per-frame degradation (the V3 domain)

`realistic_degrade_frame` applies, independently per frame: optional Gaussian blur, optional motion blur, optional Gaussian noise, optional JPEG compression, then ×4 area-downscale. **Every frame resamples its own parameters** — this is exactly the limitation V4 addresses (no temporal coherence, no real codec).

In [ ]:
def apply_jpeg_compression(img_bgr, quality=50):
    encode_param = [int(cv2.IMWRITE_JPEG_QUALITY), int(quality)]
    success, enc = cv2.imencode(".jpg", img_bgr, encode_param)
    if not success:
        return img_bgr
    return cv2.imdecode(enc, cv2.IMREAD_COLOR)

def apply_motion_blur(img_bgr, ksize=9):
    kernel = np.zeros((ksize, ksize), dtype=np.float32)
    kernel[ksize // 2, :] = 1.0
    kernel /= kernel.sum()
    return cv2.filter2D(img_bgr, -1, kernel)

def realistic_degrade_frame(
    frame_bgr, scale=4,
    blur_prob=0.6, motion_blur_prob=0.15, noise_prob=0.6, jpeg_prob=0.6,
    min_jpeg_quality=35, max_jpeg_quality=75, min_noise_std=1.0, max_noise_std=10.0,
):
    out = frame_bgr.copy()
    if random.random() < blur_prob:
        k = random.choice([3, 5])
        sigma = random.uniform(0.5, 1.6)
        out = cv2.GaussianBlur(out, (k, k), sigma)
    if random.random() < motion_blur_prob:
        out = apply_motion_blur(out, ksize=random.choice([5, 7]))
    if random.random() < noise_prob:
        noise_std = random.uniform(min_noise_std, max_noise_std)
        noise = np.random.normal(0, noise_std, out.shape).astype(np.float32)
        out = np.clip(out.astype(np.float32) + noise, 0, 255).astype(np.uint8)
    if random.random() < jpeg_prob:
        out = apply_jpeg_compression(out, quality=random.randint(min_jpeg_quality, max_jpeg_quality))
    h, w = out.shape[:2]
    return cv2.resize(out, (w // scale, h // scale), interpolation=cv2.INTER_AREA)

print("V3 per-frame degradation ready")

## II.2 — Build the V3 processed splits

Unzip REDS `train_sharp` / `val_sharp`, locate the sequence folders, and materialize `SPLIT_ROOT/{train,val,test}/<clip>/{hr_frames,lr_frames}`. Cached — reruns skip work unless `RESET_PROCESSED_DATASET` / `FORCE_REBUILD_SPLITS` are set. These `hr_frames` are also what V4 later reuses (read-only) to build its codec LR.

In [ ]:
def list_dirs(p):
    p = Path(p)
    return sorted([x for x in p.iterdir() if x.is_dir()]) if p.exists() else []

def resolve_sequence_root(base_dir, min_expected=1, max_depth=5):
    base_dir = Path(base_dir)
    def looks_like_sequence_dir(p):
        return p.is_dir() and len(list(p.glob("*.png"))) > 0
    current_level = [base_dir]
    for _ in range(max_depth + 1):
        next_level = []
        for candidate in current_level:
            children = list_dirs(candidate)
            if len(children) >= min_expected and sum(looks_like_sequence_dir(ch) for ch in children) >= min_expected:
                return candidate
            next_level.extend(children)
        current_level = next_level
    return None

def ensure_unzipped(zip_path, extract_dir):
    zip_path, extract_dir = Path(zip_path), Path(extract_dir)
    if not zip_path.exists():
        print(f"Zip not found, skipping: {zip_path}")
        return
    if extract_dir.exists() and len(list(extract_dir.rglob("*.png"))) > 0:
        print(f"Already extracted: {extract_dir}")
        return
    print(f"Extracting {zip_path} -> {extract_dir}")
    extract_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(extract_dir)

def split_cache_ready(root):
    root = Path(root)
    if not root.exists():
        return False
    for split in ["train", "val", "test"]:
        d = root / split
        if not d.exists() or len([p for p in d.iterdir() if p.is_dir()]) == 0:
            return False
    return True

def extract_from_reds_sequence(seq_dir, split_name, clip_name, scale=4, max_frames=200):
    seq_dir = Path(seq_dir)
    frame_paths = sorted(seq_dir.glob("*.png"))
    if len(frame_paths) == 0:
        raise RuntimeError(f"No PNG frames in {seq_dir}")
    if max_frames is not None:
        frame_paths = frame_paths[:max_frames]
    hr_dir = SPLIT_ROOT / split_name / clip_name / "hr_frames"
    lr_dir = SPLIT_ROOT / split_name / clip_name / "lr_frames"
    hr_dir.mkdir(parents=True, exist_ok=True)
    lr_dir.mkdir(parents=True, exist_ok=True)
    count = 0
    for frame_path in frame_paths:
        frame = cv2.imread(str(frame_path), cv2.IMREAD_COLOR)
        if frame is None:
            continue
        cv2.imwrite(str(hr_dir / f"{count:04d}.png"), frame)
        cv2.imwrite(str(lr_dir / f"{count:04d}.png"), realistic_degrade_frame(frame, scale=scale))
        count += 1
    print(f"Built {split_name}/{clip_name} from {seq_dir.name} with {count} frames")


ensure_unzipped(TRAIN_ZIP_PATH, TRAIN_SHARP_DIR)
ensure_unzipped(VAL_ZIP_PATH, VAL_SHARP_DIR)
TRAIN_SHARP_DIR = resolve_sequence_root(TRAIN_SHARP_DIR) or TRAIN_SHARP_DIR
VAL_SHARP_DIR = resolve_sequence_root(VAL_SHARP_DIR) or VAL_SHARP_DIR

train_seq_dirs = list_dirs(TRAIN_SHARP_DIR)
val_seq_dirs_all = list_dirs(VAL_SHARP_DIR)
if MAX_TRAIN_SEQS is not None:
    train_seq_dirs = train_seq_dirs[:MAX_TRAIN_SEQS]
val_seq_dirs = val_seq_dirs_all[:VAL_TO_VAL_COUNT]
test_seq_dirs = val_seq_dirs_all[VAL_TO_VAL_COUNT:VAL_TO_VAL_COUNT + VAL_TO_TEST_COUNT]
print(f"Train/Val/Test sequences: {len(train_seq_dirs)}/{len(val_seq_dirs)}/{len(test_seq_dirs)}")

if RESET_PROCESSED_DATASET and SPLIT_ROOT.exists():
    shutil.rmtree(SPLIT_ROOT)
if FORCE_REBUILD_SPLITS and SPLIT_ROOT.exists():
    shutil.rmtree(SPLIT_ROOT)

if split_cache_ready(SPLIT_ROOT):
    print("Processed splits already exist. Reusing cache.")
else:
    SPLIT_ROOT.mkdir(parents=True, exist_ok=True)
    for split_name, seqs in [("train", train_seq_dirs), ("val", val_seq_dirs), ("test", test_seq_dirs)]:
        for seq_dir in seqs:
            extract_from_reds_sequence(seq_dir, split_name, seq_dir.name, scale=SCALE, max_frames=MAX_FRAMES_PER_SEQ)

## II.3 — DataLoaders

In [ ]:
g = torch.Generator(); g.manual_seed(SEED)
_mp_context = "fork" if NUM_WORKERS > 0 else None

train_dataset = CustomVideoDataset(SPLIT_ROOT / "train", seq_len=SEQ_LEN, patch_size=PATCH_SIZE, training=True, scale=SCALE)
val_dataset = CustomVideoDataset(SPLIT_ROOT / "val", seq_len=SEQ_LEN, patch_size=PATCH_SIZE, training=False, scale=SCALE, full_frame_eval=FULL_FRAME_EVAL)
test_dataset = CustomVideoDataset(SPLIT_ROOT / "test", seq_len=SEQ_LEN, patch_size=PATCH_SIZE, training=False, scale=SCALE, full_frame_eval=FULL_FRAME_EVAL)

def _mk(ds, bs, shuffle):
    return DataLoader(ds, batch_size=bs, shuffle=shuffle, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
                      worker_init_fn=seed_worker if NUM_WORKERS > 0 else None, generator=g,
                      collate_fn=eval_collate_fn, multiprocessing_context=_mp_context)

train_loader = _mk(train_dataset, BATCH_SIZE, True)
val_loader = _mk(val_dataset, 1, False)
test_loader = _mk(test_dataset, 1, False)
print("DataLoaders ready | NUM_WORKERS:", NUM_WORKERS)

## II.4 — Model, loss, optimizer, scheduler

SPyNet gets a **lower LR** (`LR × 0.125 = 2.5e-5`) than the rest of the network — the BasicVSR recipe for a pretrained flow estimator. `ReduceLROnPlateau` halves the LR when val loss stalls.

In [ ]:
model = build_model()
criterion = CombinedRestorationLoss(charbonnier_weight=1.0, edge_weight=0.05, perceptual_weight=0.1).to(device)

spynet_params = list(model.flow_estimator.parameters())
spynet_ids = {id(p) for p in spynet_params}
other_params = [p for p in model.parameters() if id(p) not in spynet_ids]
optimizer = optim.AdamW(
    [{"params": other_params, "lr": LR},
     {"params": spynet_params, "lr": LR * 0.125}],
    weight_decay=WEIGHT_DECAY,
)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=SCHEDULER_FACTOR,
                                                 patience=SCHEDULER_PATIENCE, min_lr=MIN_LR) if USE_SCHEDULER else None
print("Main LR:", LR, "| SPyNet LR:", LR * 0.125)

## II.5 — Train V3

Resumable (`AUTO_RESUME`) from `training_state.pth`. Saves best-by-loss and best-by-PSNR checkpoints, periodic snapshots, and a `history.json` log. Runs a fast validation most epochs and a full validation every `FULL_VAL_EVERY`.

> Long GPU job — run this on the training server, not locally.

In [ ]:
def load_json_if_exists(path, default):
    path = Path(path)
    if path.exists():
        with open(path, "r") as f:
            return json.load(f)
    return default

history = load_json_if_exists(HISTORY_JSON_PATH, {
    "train_loss": [], "val_loss": [], "val_psnr": [], "val_ssim": [],
    "val_bicubic_psnr": [], "lr": [], "val_mode": [],
})
start_epoch = 0
best_val_loss = float("inf")
best_val_psnr = -float("inf")

if AUTO_RESUME and TRAINING_STATE_PATH.exists():
    print(f"Resuming from {TRAINING_STATE_PATH}")
    state = torch.load(TRAINING_STATE_PATH, map_location=device)
    model.load_state_dict(state["model_state_dict"])
    optimizer.load_state_dict(state["optimizer_state_dict"])
    if scheduler is not None and state.get("scheduler_state_dict") is not None:
        scheduler.load_state_dict(state["scheduler_state_dict"])
    start_epoch = int(state.get("epoch_completed", 0))
    best_val_loss = float(state.get("best_val_loss", best_val_loss))
    best_val_psnr = float(state.get("best_val_psnr", best_val_psnr))
    history = state.get("history", history)
    print(f"Resumed at epoch {start_epoch}/{NUM_EPOCHS}")
else:
    print("Starting fresh training")

for epoch in range(start_epoch, NUM_EPOCHS):
    current_epoch = epoch + 1
    current_lr = optimizer.param_groups[0]["lr"]

    train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device, grad_clip_norm=GRAD_CLIP_NORM)

    run_full_val = (current_epoch % FULL_VAL_EVERY == 0)
    val_metrics = evaluate_model(model, val_loader, criterion, device, scale=SCALE, shave_border=SHAVE_BORDER,
                                 max_batches=None if run_full_val else FAST_VAL_MAX_BATCHES, compute_ssim=run_full_val)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_metrics["loss"])
    history["val_psnr"].append(val_metrics["psnr"])
    history["val_ssim"].append(val_metrics["ssim"])
    history["val_bicubic_psnr"].append(val_metrics["bicubic_psnr"])
    history["lr"].append(current_lr)
    history["val_mode"].append("FULL" if run_full_val else "FAST")

    print(f"Epoch {current_epoch:03d}/{NUM_EPOCHS} | {'FULL' if run_full_val else 'FAST'} | lr={current_lr:.2e} | "
          f"train={train_loss:.4f} | val_loss={val_metrics['loss']:.4f} | val_psnr={val_metrics['psnr']:.2f} | "
          f"bicubic={val_metrics['bicubic_psnr']:.2f}", flush=True)

    if not math.isfinite(train_loss) or not math.isfinite(val_metrics["loss"]):
        print("Non-finite metric. Stopping.")
        break

    if scheduler is not None and run_full_val:
        scheduler.step(val_metrics["loss"])
    if run_full_val:
        if val_metrics["loss"] < best_val_loss:
            best_val_loss = val_metrics["loss"]
            torch.save(model.state_dict(), BEST_PATH)
            print(f"  saved best-by-loss -> {BEST_PATH.name}")
        if val_metrics["psnr"] > best_val_psnr:
            best_val_psnr = val_metrics["psnr"]
            torch.save(model.state_dict(), BEST_PSNR_PATH)
            print(f"  saved best-by-psnr -> {BEST_PSNR_PATH.name}")

    torch.save(model.state_dict(), LAST_PATH)
    torch.save({
        "epoch_completed": current_epoch, "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict() if scheduler is not None else None,
        "best_val_loss": best_val_loss, "best_val_psnr": best_val_psnr, "history": history,
    }, TRAINING_STATE_PATH)
    with open(HISTORY_JSON_PATH, "w") as f:
        json.dump(history, f, indent=2)
    if current_epoch % KEEP_EVERY_N_EPOCHS == 0:
        torch.save(model.state_dict(), EPOCH_CKPT_DIR / f"epoch_{current_epoch:03d}.pth")

## II.6 — V3 test-set evaluation & curves

In [ ]:
model.load_state_dict(torch.load(BEST_PSNR_PATH, map_location=device))
model.to(device).eval()

test_metrics = evaluate_model(model, test_loader, criterion, device, scale=SCALE, shave_border=SHAVE_BORDER, compute_ssim=True)
summary = {
    "loss": test_metrics["loss"], "psnr": test_metrics["psnr"], "ssim": test_metrics["ssim"],
    "bicubic_psnr": test_metrics["bicubic_psnr"],
    "psnr_gain_over_bicubic": test_metrics["psnr"] - test_metrics["bicubic_psnr"],
}
with open(TEST_METRICS_JSON_PATH, "w") as f:
    json.dump(summary, f, indent=2)
print(json.dumps(summary, indent=2))

if HISTORY_JSON_PATH.exists():
    with open(HISTORY_JSON_PATH) as f:
        history = json.load(f)
    plt.figure(figsize=(18, 4))
    plt.subplot(1, 3, 1); plt.plot(history["train_loss"], label="train"); plt.plot(history["val_loss"], label="val"); plt.legend(); plt.title("Loss")
    plt.subplot(1, 3, 2); plt.plot(history["val_psnr"], label="model"); plt.plot(history["val_bicubic_psnr"], label="bicubic"); plt.legend(); plt.title("Val PSNR")
    plt.subplot(1, 3, 3); plt.plot(history["lr"]); plt.title("LR")
    plt.tight_layout(); plt.show()

# Part III — V4 YouTube-codec fine-tune

The V4 stage takes the **trained V3 checkpoint** and adapts it to real compressed video, without forgetting V3's competence. Three ingredients:

1. **YouTube-style degradation** (`youtube_degrade_clip`) — samples all params **once per clip** and ends with a real `libx264` **encode/decode round-trip**, producing temporally correlated macroblocking/mosquito-noise/ringing.
2. **A fine-tune split tree** built from the V3 `hr_frames` (symlinked, read-only): `train/<clip>__codecN` (new codec LR), a **replay** subset reusing the old V3 LR (anti-forgetting), and two val sets — `val_old` (V3 degradation) and `val_codec` (new degradation).
3. **A low-LR fine-tune** (`2.5e-5`) with **SPyNet frozen**, checkpointing on the best **combined** (old+codec) val loss so the model never silently trades one domain for the other.

This mirrors `scripts/degrade_clip_video.py` + `scripts/finetune_v4_youtube.py`. Output: `vsr_model_v4_best_codec_psnr.pth` → deployed by pointing `CHECKPOINT_PATH` at it.

## III.1 — YouTube-style per-clip degradation

Per-clip pipeline (same params applied to every frame of the clip, then a whole-clip H.264 pass):

1. Gaussian blur `p=0.70` · 2. Motion blur `p=0.15` · 3. Gaussian noise `p=0.50` (per-clip std, fresh noise/frame) · 4. JPEG `p=0.15` · 5. ×4 downscale · 6. **H.264 @ CRF 23–38** · 7. optional **2nd encode** `p=0.30` (simulates upload → re-encode).

The codec round-trip is PNG → ffmpeg → PNG with **no lossy intermediate**, so the only compression is the sampled CRF. Requires **ffmpeg on PATH**.

In [ ]:
# Part III reuses apply_jpeg_compression / apply_motion_blur defined in Part II (II.1).
assert "apply_jpeg_compression" in globals() and "apply_motion_blur" in globals(), (
    "Run Part II (II.1 per-frame degradation) first: it defines "
    "apply_jpeg_compression / apply_motion_blur used here."
)

from dataclasses import dataclass

INTERPOLATIONS = {"area": cv2.INTER_AREA, "bilinear": cv2.INTER_LINEAR, "bicubic": cv2.INTER_CUBIC}

@dataclass(frozen=True)
class ClipDegradationParams:
    gaussian_blur: "tuple[int, float] | None"
    motion_blur_ksize: "int | None"
    noise_std: "float | None"
    jpeg_quality: "int | None"
    downscale_interp: str
    crf: int
    preset: str
    second_crf: "int | None"

    def describe(self):
        parts = []
        if self.gaussian_blur is not None:
            k, sigma = self.gaussian_blur; parts.append(f"blur(k={k},sigma={sigma:.2f})")
        if self.motion_blur_ksize is not None: parts.append(f"motion(k={self.motion_blur_ksize})")
        if self.noise_std is not None: parts.append(f"noise(std={self.noise_std:.1f})")
        if self.jpeg_quality is not None: parts.append(f"jpeg(q={self.jpeg_quality})")
        parts.append(f"resize({self.downscale_interp})")
        parts.append(f"h264(crf={self.crf},preset={self.preset})")
        if self.second_crf is not None: parts.append(f"h264-2nd(crf={self.second_crf})")
        return " -> ".join(parts)

def sample_clip_params(rng):
    return ClipDegradationParams(
        gaussian_blur=((rng.choice([3, 5, 7]), rng.uniform(0.4, 2.0)) if rng.random() < 0.7 else None),
        motion_blur_ksize=rng.choice([5, 7]) if rng.random() < 0.15 else None,
        noise_std=rng.uniform(1.0, 8.0) if rng.random() < 0.5 else None,
        jpeg_quality=rng.randint(50, 85) if rng.random() < 0.15 else None,
        downscale_interp=rng.choice(["area", "bilinear", "bicubic"]),
        crf=rng.randint(23, 38),
        preset=rng.choice(["veryfast", "medium"]),
        second_crf=rng.randint(28, 40) if rng.random() < 0.3 else None,
    )

def crop_to_multiple(frame_bgr, multiple):
    """Crop (top-left) so both dims divide `multiple` — needed so the ×4 LR has
    even dims for yuv420p H.264. HR is cropped to scale*2."""
    h, w = frame_bgr.shape[:2]
    new_h, new_w = (h // multiple) * multiple, (w // multiple) * multiple
    if new_h == 0 or new_w == 0:
        raise ValueError(f"Frame {w}x{h} too small for multiple {multiple}")
    return frame_bgr if (new_h == h and new_w == w) else frame_bgr[:new_h, :new_w]

def degrade_frame_pre_codec(frame_bgr, params, scale, np_rng):
    out = frame_bgr
    if params.gaussian_blur is not None:
        k, sigma = params.gaussian_blur
        out = cv2.GaussianBlur(out, (k, k), sigma)
    if params.motion_blur_ksize is not None:
        out = apply_motion_blur(out, ksize=params.motion_blur_ksize)
    if params.noise_std is not None:
        noise = np_rng.normal(0.0, params.noise_std, out.shape).astype(np.float32)
        out = np.clip(out.astype(np.float32) + noise, 0, 255).astype(np.uint8)
    if params.jpeg_quality is not None:
        out = apply_jpeg_compression(out, quality=params.jpeg_quality)
    h, w = out.shape[:2]
    return cv2.resize(out, (w // scale, h // scale), interpolation=INTERPOLATIONS[params.downscale_interp])

def run_ffmpeg(args):
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", *args], check=True)

def encode_h264(frames_dir, dest, fps, crf, preset):
    run_ffmpeg(["-framerate", f"{fps}", "-i", str(frames_dir / "%05d.png"), "-c:v", "libx264",
                "-crf", str(crf), "-preset", preset, "-pix_fmt", "yuv420p", "-movflags", "+faststart", str(dest)])

def reencode_h264(src, dest, crf, preset):
    run_ffmpeg(["-i", str(src), "-an", "-c:v", "libx264", "-crf", str(crf), "-preset", preset,
                "-pix_fmt", "yuv420p", "-movflags", "+faststart", str(dest)])

def read_video_frames(path):
    cap = cv2.VideoCapture(str(path))
    if not cap.isOpened():
        raise RuntimeError(f"Failed to open video: {path}")
    frames = []
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frames.append(frame)
    cap.release()
    return frames

def encode_lr_clip(lr_frames, params, fps, dest):
    with tempfile.TemporaryDirectory(prefix="yt_degrade_") as tmp:
        src_dir = Path(tmp) / "src"; src_dir.mkdir()
        for i, frame in enumerate(lr_frames):
            cv2.imwrite(str(src_dir / f"{i:05d}.png"), frame)
        first_pass = Path(tmp) / "pass1.mp4" if params.second_crf is not None else dest
        encode_h264(src_dir, first_pass, fps=fps, crf=params.crf, preset=params.preset)
        if params.second_crf is not None:
            reencode_h264(first_pass, dest, crf=params.second_crf, preset=params.preset)

def youtube_degrade_clip(hr_frames, scale, rng, np_rng, fps=24.0):
    """Degrade one clip of HR BGR frames -> (cropped_hr, codec_lr, params).
    Save the RETURNED HR (it may be cropped for codec-safe dims), not the originals."""
    if len(hr_frames) == 0:
        raise ValueError("Empty clip")
    params = sample_clip_params(rng)
    hr_out = [crop_to_multiple(f, scale * 2) for f in hr_frames]
    lr_pre = [degrade_frame_pre_codec(f, params, scale, np_rng) for f in hr_out]
    with tempfile.TemporaryDirectory(prefix="yt_degrade_") as tmp:
        encoded = Path(tmp) / "clip.mp4"
        encode_lr_clip(lr_pre, params, fps=fps, dest=encoded)
        lr_frames = read_video_frames(encoded)
    if len(lr_frames) != len(hr_out):
        raise RuntimeError(f"Codec round-trip changed frame count: {len(hr_out)} -> {len(lr_frames)}")
    return hr_out, lr_frames, params

assert shutil.which("ffmpeg") is not None, "ffmpeg required on PATH for V4 codec degradation"
print("YouTube codec degradation ready")

## III.2 — V4 config & split builder

`V4` holds the fine-tune knobs (mirroring `finetune_v4_youtube.py`'s CLI defaults). The split builder reads the V3 `hr_frames` **read-only** (symlinked) and generates codec LR variants + a replay subset + `val_old`/`val_codec`. A per-clip `done.txt` marker makes it idempotent/resumable.

In [ ]:
from types import SimpleNamespace

V4 = SimpleNamespace(
    v3_splits=SPLIT_ROOT,
    workspace=WORK_ROOT / "experiments" / "model_v4_finetune_youtube",
    init_weights=BEST_PSNR_PATH,          # the V3 checkpoint to fine-tune from
    epochs=30,
    batch_size=4,
    lr=2.5e-5,                            # V3 trained at 2e-4
    seq_len=SEQ_LEN,
    patch_size=PATCH_SIZE,
    scale=SCALE,
    variants=2,                          # codec-degraded variants per train clip
    replay_ratio=0.25,                   # fraction of train data using old V3 LR
    num_workers=8,
    seed=SEED,
    max_clips=None,                      # subset for smoke tests
    unfreeze_spynet=False,               # default: SPyNet frozen (forgetting insurance)
)

def clip_seed(base_seed, key):
    return int(hashlib.sha256(f"{base_seed}:{key}".encode()).hexdigest()[:8], 16)

def link_or_copy(src, dest):
    if dest.exists() or dest.is_symlink():
        return
    try:
        dest.symlink_to(src, target_is_directory=True)
    except OSError:
        shutil.copytree(src, dest)

def read_frames(frames_dir):
    frames = [cv2.imread(str(p), cv2.IMREAD_COLOR) for p in sorted(Path(frames_dir).glob("*.png"))]
    frames = [f for f in frames if f is not None]
    if len(frames) == 0:
        raise RuntimeError(f"No readable PNG frames in {frames_dir}")
    return frames

def build_codec_clip(v3_clip, dest_clip, seed_key, base_seed):
    """One codec view: symlinked hr_frames + new H.264 LR frames."""
    if (dest_clip / "done.txt").exists():
        return
    dest_clip.mkdir(parents=True, exist_ok=True)
    link_or_copy(v3_clip / "hr_frames", dest_clip / "hr_frames")
    seed = clip_seed(base_seed, seed_key)
    rng, np_rng = random.Random(seed), np.random.default_rng(seed)
    hr_frames = read_frames(v3_clip / "hr_frames")
    hr_out, lr_frames, params = youtube_degrade_clip(hr_frames, scale=V4.scale, rng=rng, np_rng=np_rng)
    if len(hr_out) != len(hr_frames) or hr_out[0].shape != hr_frames[0].shape:
        # HR needed cropping -> materialize cropped HR so LR/HR stay aligned.
        hr_link = dest_clip / "hr_frames"
        (hr_link.unlink() if hr_link.is_symlink() else shutil.rmtree(hr_link))
        hr_dir = dest_clip / "hr_frames"; hr_dir.mkdir()
        for i, frame in enumerate(hr_out):
            cv2.imwrite(str(hr_dir / f"{i:04d}.png"), frame)
    lr_dir = dest_clip / "lr_frames"
    if lr_dir.exists():
        shutil.rmtree(lr_dir)
    lr_dir.mkdir()
    for i, frame in enumerate(lr_frames):
        cv2.imwrite(str(lr_dir / f"{i:04d}.png"), frame)
    (dest_clip / "done.txt").write_text(params.describe() + "\n")
    print(f"  built {dest_clip.parent.name}/{dest_clip.name}: {params.describe()}", flush=True)

def build_replay_clip(v3_clip, dest_clip):
    """Replay view: both hr_frames and the OLD V3 lr_frames, symlinked."""
    if (dest_clip / "done.txt").exists():
        return
    dest_clip.mkdir(parents=True, exist_ok=True)
    link_or_copy(v3_clip / "hr_frames", dest_clip / "hr_frames")
    link_or_copy(v3_clip / "lr_frames", dest_clip / "lr_frames")
    (dest_clip / "done.txt").write_text("replay: V3 lr_frames (old per-frame degradation)\n")

def list_clip_dirs(root):
    root = Path(root)
    return sorted([p for p in root.iterdir() if p.is_dir()]) if root.exists() else []

def build_v4_splits():
    v3_splits = Path(V4.v3_splits)
    v4_splits = Path(V4.workspace) / "splits"
    train_clips = list_clip_dirs(v3_splits / "train")
    val_clips = list_clip_dirs(v3_splits / "val")
    if len(train_clips) == 0 or len(val_clips) == 0:
        raise RuntimeError(f"V3 splits not found under {v3_splits} — run Part II first.")
    if V4.max_clips is not None:
        train_clips = train_clips[:V4.max_clips]
        val_clips = val_clips[:max(1, V4.max_clips // 2)]
    print(f"Building V4 splits at {v4_splits}\n  from {len(train_clips)} train / {len(val_clips)} val V3 clips")

    for clip in train_clips:
        for variant in range(1, V4.variants + 1):
            build_codec_clip(clip, v4_splits / "train" / f"{clip.name}__codec{variant}",
                             seed_key=f"train/{clip.name}/v{variant}", base_seed=V4.seed)

    codec_count = len(train_clips) * V4.variants
    replay_count = min(round(V4.replay_ratio / (1.0 - V4.replay_ratio) * codec_count), len(train_clips))
    replay_rng = random.Random(clip_seed(V4.seed, "replay-selection"))
    for clip in replay_rng.sample(train_clips, replay_count):
        build_replay_clip(clip, v4_splits / "train" / f"{clip.name}__replay")
    print(f"  replay clips: {replay_count} (target ratio {V4.replay_ratio})")

    for clip in val_clips:
        build_codec_clip(clip, v4_splits / "val_codec" / clip.name, seed_key=f"val/{clip.name}", base_seed=V4.seed)
        build_replay_clip(clip, v4_splits / "val_old" / clip.name)
    return v4_splits

V4_SPLITS = build_v4_splits()
print("V4 splits ready at", V4_SPLITS)

## III.3 — Fine-tune V4

Load the V3 weights, **freeze SPyNet**, and fine-tune at `2.5e-5` with the same combined loss. Each epoch reports PSNR on **both** val sets vs the V3 baseline: codec PSNR should climb while old PSNR holds (a drop >0.3 dB warns of forgetting). Checkpoints on best combined loss and best codec PSNR.

In [ ]:
val_old_ds = CustomVideoDataset(V4_SPLITS / "val_old", seq_len=V4.seq_len, patch_size=V4.patch_size, training=False, scale=V4.scale)
val_codec_ds = CustomVideoDataset(V4_SPLITS / "val_codec", seq_len=V4.seq_len, patch_size=V4.patch_size, training=False, scale=V4.scale)
v4_train_ds = CustomVideoDataset(V4_SPLITS / "train", seq_len=V4.seq_len, patch_size=V4.patch_size, training=True, scale=V4.scale)

def _v4_loader(ds, bs, shuffle):
    return DataLoader(ds, batch_size=bs, shuffle=shuffle, num_workers=V4.num_workers, pin_memory=True,
                      worker_init_fn=seed_worker if V4.num_workers > 0 else None,
                      collate_fn=eval_collate_fn,
                      multiprocessing_context="fork" if V4.num_workers > 0 else None)

v4_train_loader = _v4_loader(v4_train_ds, V4.batch_size, True)
val_old_loader = _v4_loader(val_old_ds, 1, False)
val_codec_loader = _v4_loader(val_codec_ds, 1, False)

# ── model: start from V3 weights, freeze SPyNet ──────────────────────────────
v4_model = build_model()
assert V4.init_weights.exists(), (
    f"V3 init checkpoint not found: {V4.init_weights}. "
    "Run Part II (V3 training) first, or point V4.init_weights at an existing .pth."
)
sd = torch.load(V4.init_weights, map_location="cpu")
sd = sd["model_state_dict"] if "model_state_dict" in sd else sd
v4_model.load_state_dict(sd)
v4_model.to(device)
print(f"Loaded V3 weights from {V4.init_weights}")

if V4.unfreeze_spynet:
    spynet_params = list(v4_model.flow_estimator.parameters())
    spynet_ids = {id(p) for p in spynet_params}
    other = [p for p in v4_model.parameters() if id(p) not in spynet_ids]
    v4_optimizer = optim.AdamW([{"params": other, "lr": V4.lr},
                                {"params": spynet_params, "lr": V4.lr * 0.1}], weight_decay=1e-4)
    print(f"SPyNet UNFROZEN at lr {V4.lr * 0.1:.2e}")
else:
    v4_model.flow_estimator.requires_grad_(False)
    v4_optimizer = optim.AdamW([p for p in v4_model.parameters() if p.requires_grad], lr=V4.lr, weight_decay=1e-4)
    print("SPyNet frozen")

v4_criterion = CombinedRestorationLoss(1.0, 0.05, 0.1).to(device)
v4_scheduler = optim.lr_scheduler.ReduceLROnPlateau(v4_optimizer, mode="min", factor=0.5, patience=3, min_lr=1e-6)

V4_SAVE = Path(V4.workspace) / "checkpoints"; V4_SAVE.mkdir(parents=True, exist_ok=True)
V4_LOGS = Path(V4.workspace) / "logs"; V4_LOGS.mkdir(parents=True, exist_ok=True)
best_combined_path = V4_SAVE / "vsr_model_v4_best_combined.pth"
best_codec_psnr_path = V4_SAVE / "vsr_model_v4_best_codec_psnr.pth"

# ── V3 baseline on both val sets (the reference we must not regress on) ───────
base_old = evaluate_model(v4_model, val_old_loader, v4_criterion, device, scale=V4.scale, shave_border=V4.scale)
base_codec = evaluate_model(v4_model, val_codec_loader, v4_criterion, device, scale=V4.scale, shave_border=V4.scale)
print(f"V3 baseline | old psnr={base_old['psnr']:.2f} | codec psnr={base_codec['psnr']:.2f} "
      f"(bicubic {base_codec['bicubic_psnr']:.2f})")

best_combined = float("inf")
best_codec_psnr = -float("inf")
v4_history = {"train_loss": [], "val_old_psnr": [], "val_codec_psnr": [], "combined_loss": [], "lr": []}

for epoch in range(V4.epochs):
    current_lr = v4_optimizer.param_groups[0]["lr"]
    train_loss = train_one_epoch(v4_model, v4_train_loader, v4_optimizer, v4_criterion, device, grad_clip_norm=1.0)
    val_old = evaluate_model(v4_model, val_old_loader, v4_criterion, device, scale=V4.scale, shave_border=V4.scale)
    val_codec = evaluate_model(v4_model, val_codec_loader, v4_criterion, device, scale=V4.scale, shave_border=V4.scale)
    combined = 0.5 * (val_old["loss"] + val_codec["loss"])

    v4_history["train_loss"].append(train_loss)
    v4_history["val_old_psnr"].append(val_old["psnr"])
    v4_history["val_codec_psnr"].append(val_codec["psnr"])
    v4_history["combined_loss"].append(combined)
    v4_history["lr"].append(current_lr)

    old_drop = base_old["psnr"] - val_old["psnr"]
    print(f"Epoch {epoch+1:03d}/{V4.epochs} | lr={current_lr:.2e} | train={train_loss:.4f} | "
          f"old psnr={val_old['psnr']:.2f} ({-old_drop:+.2f} vs V3) | "
          f"codec psnr={val_codec['psnr']:.2f} ({val_codec['psnr']-base_codec['psnr']:+.2f} vs V3) | "
          f"combined={combined:.4f}", flush=True)
    if old_drop > 0.3:
        print(f"  WARNING: old-val PSNR dropped {old_drop:.2f} dB — forgetting; raise replay or lower lr")

    if not math.isfinite(train_loss) or not math.isfinite(combined):
        print("Non-finite metric. Stopping."); break

    v4_scheduler.step(combined)
    if combined < best_combined:
        best_combined = combined
        torch.save(v4_model.state_dict(), best_combined_path)
        print(f"  saved best-combined -> {best_combined_path.name}")
    if val_codec["psnr"] > best_codec_psnr:
        best_codec_psnr = val_codec["psnr"]
        torch.save(v4_model.state_dict(), best_codec_psnr_path)
        print(f"  saved best-codec-psnr -> {best_codec_psnr_path.name}")
    with open(V4_LOGS / "history.json", "w") as f:
        json.dump({"baseline": {"old": base_old, "codec": base_codec}, "history": v4_history}, f, indent=2)

print(f"\nDone. best combined={best_combined:.4f} | best codec psnr={best_codec_psnr:.2f}")
print(f"Deploy candidate: {best_codec_psnr_path}")
print("To use in the app: copy it to apps/ai/checkpoints/ and point CHECKPOINT_PATH at it.")

# Part IV — Inference & deploy

The production path: take a **real** video (already compressed — no synthetic downscale), run the model over a **15-frame sliding window** centered on each output frame (edge windows padded with the last frame), and write the ×4 result. This is the same sliding-window logic implemented in `apps/ai/baseline/vsr_inference.py` and served by the FastAPI inference service.

Point `CHECKPOINT` at whichever weights you want to run — the V4 codec checkpoint for real-world footage, the V3 checkpoint for the clean-degradation baseline.

In [ ]:
# ── choose the checkpoint to run ─────────────────────────────────────────────
CHECKPOINT = best_codec_psnr_path if 'best_codec_psnr_path' in dir() and best_codec_psnr_path.exists() else BEST_PSNR_PATH
INPUT_VIDEO = "/path/to/input.mp4"      # <-- set to a real video
OUTPUT_DIR = RESULTS_DIR / "inference_demo"
MAX_FRAMES = None
MAX_LR_HEIGHT = 480                     # downscale only if taller (GPU memory guard)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
infer_model = build_model()
infer_model.load_state_dict(torch.load(CHECKPOINT, map_location=device))
infer_model.to(device).eval()
print("Loaded checkpoint:", CHECKPOINT)

cap = cv2.VideoCapture(INPUT_VIDEO)
fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
ret, first_frame = cap.read()
if not ret:
    raise RuntimeError(f"Could not read {INPUT_VIDEO}")
cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
orig_h, orig_w = first_frame.shape[:2]

if orig_h > MAX_LR_HEIGHT:
    scale_factor = MAX_LR_HEIGHT / orig_h
    lr_w, lr_h, need_resize = int(orig_w * scale_factor), MAX_LR_HEIGHT, True
else:
    lr_w, lr_h, need_resize = orig_w, orig_h, False
print(f"Input {orig_w}x{orig_h} @ {fps:.1f}fps -> LR {lr_w}x{lr_h} -> SR {lr_w*4}x{lr_h*4}")

frames_bgr_lr = []
while True:
    ret, frame = cap.read()
    if not ret:
        break
    frames_bgr_lr.append(cv2.resize(frame, (lr_w, lr_h), interpolation=cv2.INTER_AREA) if need_resize else frame)
    if MAX_FRAMES and len(frames_bgr_lr) >= MAX_FRAMES:
        break
cap.release()
print(f"Extracted {len(frames_bgr_lr)} frames")

def frame_to_tensor(frame_bgr):
    return torch.from_numpy(cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB).transpose(2, 0, 1)).float() / 255.0

lr_tensors = [frame_to_tensor(f) for f in frames_bgr_lr]
half = SEQ_LEN // 2
output_frames = [None] * len(lr_tensors)

with torch.no_grad():
    for center in range(len(lr_tensors)):
        start = max(0, center - half)
        end = min(len(lr_tensors), start + SEQ_LEN)
        start = max(0, end - SEQ_LEN)
        window = torch.stack(lr_tensors[start:end], dim=0).unsqueeze(0).to(device)
        if window.shape[1] < SEQ_LEN:  # pad short edge windows with the last frame
            pad = window[:, -1:].repeat(1, SEQ_LEN - window.shape[1], 1, 1, 1)
            window = torch.cat([window, pad], dim=1)
        pred = infer_model(window)
        output_frames[center] = pred[0, center - start].cpu()
        del pred, window
        if device.type == "cuda":
            torch.cuda.empty_cache()
        if (center + 1) % 30 == 0 or center == len(lr_tensors) - 1:
            print(f"Processed {center + 1}/{len(lr_tensors)}")

# ── write output frames + video (mp4v; the service re-encodes to H.264) ──────
sr_dir = OUTPUT_DIR / "sr_frames"; sr_dir.mkdir(exist_ok=True)
sr_h, sr_w = output_frames[0].shape[1], output_frames[0].shape[2]
for i, t in enumerate(output_frames):
    cv2.imwrite(str(sr_dir / f"{i:04d}.png"), cv2.cvtColor(tensor_to_img(t), cv2.COLOR_RGB2BGR))

writer = cv2.VideoWriter(str(OUTPUT_DIR / "sr.mp4"), cv2.VideoWriter_fourcc(*"mp4v"), fps, (sr_w, sr_h))
for p in sorted(sr_dir.glob("*.png")):
    writer.write(cv2.imread(str(p)))
writer.release()
print("Wrote", OUTPUT_DIR / "sr.mp4", f"({sr_w}x{sr_h})")

# ── before/after preview ─────────────────────────────────────────────────────
idx = len(frames_bgr_lr) // 2
before = cv2.cvtColor(cv2.resize(frames_bgr_lr[idx], (sr_w, sr_h), interpolation=cv2.INTER_CUBIC), cv2.COLOR_BGR2RGB)
after = tensor_to_img(output_frames[idx])
plt.figure(figsize=(14, 6))
plt.subplot(1, 2, 1); plt.imshow(before); plt.title(f"Input (bicubic {orig_w}x{orig_h})"); plt.axis("off")
plt.subplot(1, 2, 2); plt.imshow(after); plt.title(f"VSR output ({sr_w}x{sr_h})"); plt.axis("off")
plt.tight_layout(); plt.show()

## Deploying the checkpoint

The FastAPI inference service (`apps/ai/server.py`) loads whichever `.pth` `CHECKPOINT_PATH` points to and reconstructs `BasicVSRRecurrentSeq` from `apps/ai/baseline/model_architecture.py` (the same class defined in Part I).

```bash
# V4 codec fine-tune (default for real-world footage) already ships as:
#   apps/ai/checkpoints/vsr_model_v4_best_codec_psnr.pth
# Select it:
CHECKPOINT_PATH=./checkpoints/vsr_model_v4_best_codec_psnr.pth pnpm --filter ai dev

# Verify which weights are live:
curl localhost:8000/health   # -> {"checkpoint": "vsr_model_v4_best_codec_psnr.pth", "model_loaded": true, ...}
```

> **Keep in sync.** If the architecture in Part I changes, update `apps/ai/baseline/model_architecture.py`; if the degradation/fine-tune changes, update `scripts/degrade_clip_video.py` / `scripts/finetune_v4_youtube.py`. The docs mandate (`apps/ai/AGENTS.md`) treats drift between this notebook and the shipped code as a bug.